# PharmaLens AI — Smart Target Planning v1

**Purpose:** build an explainable next-year target plan in **Units + Value**, supporting **Top-Down, Bottom-Up and Iterative** planning.

### Data rules
- IMS/Sales is the baseline.
- Market Share is calculated from Sales/IMS.
- ATC4 is already the available molecule/therapeutic classification field.
- Territory % is **not** multiplied by an individual product target.
- RX is aligned to its available period before opportunity scoring.
- Regional product sales are never invented when unavailable.


In [ ]:
from pathlib import Path
import sys, json, pandas as pd, numpy as np

ROOT = Path.cwd()
MODULE_DIR = ROOT
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from data_sources import (
    load_sales, load_processed_parquet, load_rx,
    load_territory_map, load_am_potentiality,
    annual_sales, sales_market_share
)
from target_engine import (
    company_target, build_product_targets,
    build_region_opportunity, top_down,
    bottom_up, iterative_reconcile, explain_target
)

TARGET_YEAR = 2027
BASE_YEAR = 2026
OBJECTIVE = "growth"          # growth | market_share | market_capture | custom
METHOD = "iterative"          # top_down | bottom_up | iterative
GROWTH_RATE = 0.10

print("Configuration loaded.")


In [ ]:
# Resolve project data paths.
# The notebook works with either the processed parquet or the original IMS workbook.

CANDIDATES = {
    "processed_sales": ROOT / "data" / "processed" / "cleaned_pharma_data.parquet",
    "ims": ROOT / "IMS (2021-2025).xlsx",
    "rx": ROOT / "Rx Sep-2025.xlsx",
    "territory": ROOT / "EGYPT TERR - IMS BRICKS.xls",
    "am": ROOT / "Cairo & Giza AM.xlsx",
}

for k, p in CANDIDATES.items():
    print(f"{k:16} {'FOUND' if p.exists() else 'MISSING'}  {p}")


In [ ]:
# Load the canonical sales source.
if CANDIDATES["processed_sales"].exists():
    sales = load_processed_parquet(CANDIDATES["processed_sales"])
elif CANDIDATES["ims"].exists():
    sales = load_sales(CANDIDATES["ims"])
else:
    raise FileNotFoundError(
        "IMS/Sales source not found. Place the existing cleaned parquet at "
        "data/processed/cleaned_pharma_data.parquet or the original IMS workbook "
        "at the project root."
    )

print("Sales shape:", sales.shape)
print("Years:", sorted(pd.to_numeric(sales["Year"], errors="coerce").dropna().unique())[-10:])
print("Columns:", list(sales.columns))


In [ ]:
# Calculate annual baseline and derived Market Share.
annual = annual_sales(sales)
share = sales_market_share(sales)

print("Annual rows:", len(annual))
display(annual.head(10))


In [ ]:
# Load RX.
rx = None
if CANDIDATES["rx"].exists():
    rx = load_rx(CANDIDATES["rx"])
    print("RX shape:", rx.shape)
    print("RX columns:", list(rx.columns))
    display(rx.head(10))
else:
    print("RX workbook not found — regional opportunity will remain unavailable.")


In [ ]:
# Load territory brick map and AM execution-potential context.
territory_raw = None
am_raw = None

if CANDIDATES["territory"].exists():
    territory_raw, _ = load_territory_map(CANDIDATES["territory"])
    print("Territory raw shape:", territory_raw.shape)

if CANDIDATES["am"].exists():
    am_raw = load_am_potentiality(CANDIDATES["am"])
    print("AM sheets:", list(am_raw.keys()))
    for name, frame in am_raw.items():
        print(name, frame.shape)


## 1. Company target

The company target is created first. Regional allocation must reconcile back to this exact company target.


In [ ]:
company = company_target(
    sales=sales,
    base_year=BASE_YEAR,
    target_year=TARGET_YEAR,
    objective=OBJECTIVE,
    growth_rate=GROWTH_RATE
)
company


## 2. Product target

Historical product performance is used to establish a product-level target. The molecule/therapeutic field comes from the existing IMS `ATC4` → `Therapeutic Class` mapping.


In [ ]:
product_targets = build_product_targets(
    sales, BASE_YEAR, TARGET_YEAR,
    growth_rate=GROWTH_RATE,
    objective=OBJECTIVE
)
display(product_targets.head(20))
print("Product target units:", product_targets["Target_Units"].sum())
print("Product target value:", product_targets["Target_Value"].sum())


## 3. Regional Opportunity

RX provides region/product/molecule demand signals where available. We do **not** apply a territory percentage directly to each product target.


In [ ]:
if rx is not None:
    opportunity = build_region_opportunity(rx)
    display(opportunity.head(20))
else:
    opportunity = pd.DataFrame()


In [ ]:
if not opportunity.empty:
    regional_top_down = top_down(company, opportunity)
    regional_top_down["Target_Rationale"] = regional_top_down.apply(explain_target, axis=1)
    display(regional_top_down.head(20))
    print("Regional units:", regional_top_down["Target_Units"].sum())
    print("Regional value:", regional_top_down["Target_Value"].sum())
else:
    regional_top_down = pd.DataFrame()
    print("No regional opportunity dataset available.")


## 4. Iterative reconciliation

The iterative method forces regional targets back to the exact company target while preserving relative opportunity weights.


In [ ]:
if not regional_top_down.empty:
    regional_final = iterative_reconcile(company, regional_top_down)
    display(regional_final.head(20))
    print("Final units:", regional_final["Target_Units"].sum(), "vs", company["target_units"])
    print("Final value:", regional_final["Target_Value"].sum(), "vs", company["target_value"])
else:
    regional_final = pd.DataFrame()


## 5. QA / management checks

A plan is considered structurally valid only when:
- Units and Value are non-negative.
- Regional totals reconcile to company totals.
- No product target is allocated using a generic territory percentage.
- Missing regional data is disclosed rather than fabricated.


In [ ]:
checks = {
    "company_target_units": company["target_units"],
    "company_target_value": company["target_value"],
    "regional_reconciles_units": (
        bool(np.isclose(regional_final["Target_Units"].sum(), company["target_units"]))
        if not regional_final.empty else None
    ),
    "regional_reconciles_value": (
        bool(np.isclose(regional_final["Target_Value"].sum(), company["target_value"]))
        if not regional_final.empty else None
    ),
    "no_negative_product_units": bool((product_targets["Target_Units"] >= 0).all()),
    "no_negative_product_value": bool((product_targets["Target_Value"] >= 0).all()),
}
checks


In [ ]:
# Export.
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
output_file = OUTPUT_DIR / f"PharmaLens_Target_Plan_{TARGET_YEAR}.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    pd.DataFrame([company]).to_excel(writer, sheet_name="Company_Target", index=False)
    product_targets.to_excel(writer, sheet_name="Product_Targets", index=False)
    if not opportunity.empty:
        opportunity.to_excel(writer, sheet_name="Opportunity", index=False)
    if not regional_final.empty:
        regional_final.to_excel(writer, sheet_name="Regional_Targets", index=False)
    pd.DataFrame([checks]).to_excel(writer, sheet_name="QA", index=False)

print("Exported:", output_file)


## API integration contract

The reusable backend logic is in:
- `data_sources.py`
- `target_engine.py`
- `api_router.py`

Mount the router into the **existing FastAPI app** after inspecting existing routes. Do not create a duplicate FastAPI application or duplicate endpoint.

Endpoints:
- `GET /api/v1/targets/health`
- `POST /api/v1/targets/validate`
- `POST /api/v1/targets/plan`
- `POST /api/v1/targets/opportunity`

The frontend/Copilot should consume structured outputs from these endpoints rather than recalculating or inventing target numbers.
